# THOP and activity-aware MAC profiling for `snn.py`

This notebook reports two different notions of compute for `WrapCUBASpikingCNN` in `eegmmidb/snn.py`:

- `dense_macs_thop`: standard THOP MACs from tensor shapes
- `effective_synaptic_macs`: activity-aware MACs estimated from actual nonzero activations and spikes on real inputs

Important distinction:
- The network still iterates over all `spike_ts` timesteps in code.
- Therefore you should not replace the time dimension with the number of spikes.
- Instead, for real EEG input, this notebook measures actual spike counts and estimates the effective synaptic work caused by those spikes.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

cwd = Path.cwd().resolve()
print(cwd)
# reorganized_repo_2
REPO_ROOT = cwd.parent.parent
print(f"Repository root: {REPO_ROOT}")
# Correct paths
NOTEBOOK_ROOT = cwd
THOP_ROOT = NOTEBOOK_ROOT / "thop"
DATASET_ROOT = REPO_ROOT / "dataset"
SNN_ROOT = REPO_ROOT / "src" / "models"

# Add to Python import path
sys.path.insert(0, str(THOP_ROOT))

print(f"THOP root: {THOP_ROOT}")

sys.path.insert(0, str(DATASET_ROOT))
sys.path.insert(0, str(SNN_ROOT))

from thop import clever_format, profile
from thop.vision.basic_hooks import zero_ops

from snn import (
    WrapCUBASpikingCNN,
    FeedForwardCUBALIFCell,
    FeedForwardCUBALIFCellDropout,
    RecurrentCUBALIFCell,
    TemporalConvCUBALIFCell,
)

from dataset import EEGDataset2DLeftRight, ToTensor


print(f"THOP root: {THOP_ROOT}")

/scratch/Anxiong/EEG-SNN-updated/reorganized_repo_2/notebooks/MAC
Repository root: /scratch/Anxiong/EEG-SNN-updated/reorganized_repo_2
THOP root: /scratch/Anxiong/EEG-SNN-updated/reorganized_repo_2/notebooks/MAC/thop
THOP root: /scratch/Anxiong/EEG-SNN-updated/reorganized_repo_2/notebooks/MAC/thop


In [2]:
SPIKE_TS = 160
PARAM_LIST = [0.1, 0.1, 0.1, 0.3, 0.01, 0.1, 0.01]
BATCH_SIZE = 1
HEIGHT = 10
WIDTH = 11
CHANNELS = 1
SUBJECT_ID = 1

# Set the pretrained checkpoint path here.
PRETRAINED_SNN_PATH = Path(REPO_ROOT / "snn_experiment_results/SSTL_pretrained_model/model_online_seed_5_fold_1_dut_dut2_thr_0p025_thrasym_1p0_Ascale_1p0_Aasym_1p0_rsdc2c_0.0_20251127_215126.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = WrapCUBASpikingCNN(spike_ts=SPIKE_TS, device=device, param_list=PARAM_LIST).to(device)

if not PRETRAINED_SNN_PATH.exists():
    raise FileNotFoundError(f"Pretrained SNN checkpoint not found: {PRETRAINED_SNN_PATH}")

checkpoint = torch.load(PRETRAINED_SNN_PATH, map_location=device)
state_dict = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint
state_dict = {
    (key[7:] if key.startswith("module.") else key): value
    for key, value in state_dict.items()
}
model.load_state_dict(state_dict, strict=True)
model.eval()

custom_ops = {torch.nn.Identity: zero_ops}
exact_params = sum(p.numel() for p in model.parameters())

def format_count(value):
    formatted = clever_format([float(value)], "%.3f")
    return formatted[0] if isinstance(formatted, (list, tuple)) else formatted

def profile_once(input_tensor, ret_layer_info=False):
    return profile(
        model,
        inputs=(input_tensor,),
        custom_ops=custom_ops,
        verbose=False,
        ret_layer_info=ret_layer_info,
    )

def flatten_layer_info(tree, prefix=""):
    rows = []
    for name, (ops, params, children) in tree.items():
        full_name = f"{prefix}.{name}" if prefix else name
        rows.append({"layer": full_name, "macs": float(ops), "params": float(params)})
        rows.extend(flatten_layer_info(children, full_name))
    return rows

print(model.__class__.__name__)
print(f"Device: {device}")
print(f"Loaded checkpoint: {PRETRAINED_SNN_PATH}")
print(f"Exact parameters: {exact_params:,} ({format_count(exact_params)})")


WrapCUBASpikingCNN
Device: cuda
Loaded checkpoint: /scratch/Anxiong/EEG-SNN-updated/reorganized_repo_2/snn_experiment_results/SSTL_pretrained_model/model_online_seed_5_fold_1_dut_dut2_thr_0p025_thrasym_1p0_Ascale_1p0_Aasym_1p0_rsdc2c_0.0_20251127_215126.pth
Exact parameters: 721,312 (721.312K)


/tmp/ipykernel_913925/4201130751.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PRETRAINED_SNN_PATH, map_location=device)


## Dense THOP profile on a dummy input

This is the architecture-level MAC estimate. It depends on model structure and tensor shapes, not on the actual values in the tensor.


In [3]:
dummy_input = torch.randn(BATCH_SIZE, CHANNELS, HEIGHT, WIDTH, SPIKE_TS, device=device)
dummy_macs, dummy_thop_params, dummy_layer_info = profile_once(dummy_input, ret_layer_info=True)

dummy_summary = pd.DataFrame([
    {
        "input": "dummy",
        "shape": str(tuple(dummy_input.shape)),
        "dense_macs_thop": float(dummy_macs),
        "dense_macs_fmt": format_count(dummy_macs),
        "thop_params": float(dummy_thop_params),
        "thop_params_fmt": format_count(dummy_thop_params),
        "exact_params": float(exact_params),
        "exact_params_fmt": format_count(exact_params),
    }
])
dummy_summary


,input,shape,dense_macs_thop,dense_macs_fmt,thop_params,thop_params_fmt,exact_params,exact_params_fmt
0,dummy,"(1, 1, 10, 11, 160)",601772032.0,601.772M,699136.0,699.136K,721312.0,721.312K


In [4]:
layer_df = pd.DataFrame(flatten_layer_info(dummy_layer_info))
layer_df["macs_fmt"] = layer_df["macs"].map(format_count)
layer_df["params_fmt"] = layer_df["params"].map(format_count)
layer_df.sort_values("macs", ascending=False).reset_index(drop=True)


,layer,macs,params,macs_fmt,params_fmt
0,snn,601772032.0,699136.0,601.772M,699.136K
1,snn.conv2.psp_func,495452160.0,73856.0,495.452M,73.856K
2,snn.conv2,495452160.0,73856.0,495.452M,73.856K
3,snn.conv3,47185920.0,295168.0,47.186M,295.168K
4,snn.conv3.psp_func,47185920.0,295168.0,47.186M,295.168K
5,snn.temp_conv1.psp_func_list,31260672.0,197376.0,31.261M,197.376K
6,snn.temp_conv1,31260672.0,197376.0,31.261M,197.376K
7,snn.fc1.psp_func,10485760.0,65792.0,10.486M,65.792K
8,snn.rec1.rec_func,10485760.0,65792.0,10.486M,65.792K
9,snn.rec1,10485760.0,65792.0,10.486M,65.792K


## Activity-aware helpers

The functions below estimate effective synaptic MACs from actual nonzero activations:
- `Conv2d`: count nonzero entries in the unfolded receptive-field patches and multiply by the number of connected output channels
- `Linear`: count nonzero input activations and multiply by `out_features`

This is different from THOP. THOP counts the dense operation implied by shape even if many spikes are zero.


In [5]:
SPIKE_CELL_TYPES = (
    FeedForwardCUBALIFCell,
    FeedForwardCUBALIFCellDropout,
    RecurrentCUBALIFCell,
    TemporalConvCUBALIFCell,
)

def conv2d_dense_macs(module, x, y):
    batch = y.shape[0]
    out_h = y.shape[2]
    out_w = y.shape[3]
    kernel_mul = (module.in_channels // module.groups) * module.kernel_size[0] * module.kernel_size[1]
    return float(batch * module.out_channels * out_h * out_w * kernel_mul)

def conv2d_effective_macs(module, x):
    x0 = x[0].detach()
    unfolded = F.unfold(
        x0,
        kernel_size=module.kernel_size,
        dilation=module.dilation,
        padding=module.padding,
        stride=module.stride,
    )
    active_patch_entries = unfolded.ne(0).sum().item()
    outputs_per_entry = module.out_channels // module.groups
    return float(active_patch_entries * outputs_per_entry)

def linear_dense_macs(module, x, y):
    return float(y.numel() * module.in_features)

def linear_effective_macs(module, x):
    x0 = x[0].detach()
    active_inputs = x0.ne(0).sum().item()
    return float(active_inputs * module.out_features)

def trace_activity_and_effective_macs(input_tensor):
    synaptic_rows = []
    spike_rows = []
    handles = []

    def make_synaptic_hook(name, module):
        def hook(mod, x, y):
            if isinstance(mod, nn.Conv2d):
                dense = conv2d_dense_macs(mod, x, y)
                effective = conv2d_effective_macs(mod, x)
            elif isinstance(mod, nn.Linear):
                dense = linear_dense_macs(mod, x, y)
                effective = linear_effective_macs(mod, x)
            else:
                return

            x0 = x[0].detach()
            total_inputs = x0.numel()
            active_inputs = x0.ne(0).sum().item()
            synaptic_rows.append(
                {
                    "layer": name,
                    "module_type": type(mod).__name__,
                    "dense_macs": float(dense),
                    "effective_macs": float(effective),
                    "savings_ratio": 0.0 if dense == 0 else 1.0 - (effective / dense),
                    "active_inputs": float(active_inputs),
                    "total_inputs": float(total_inputs),
                    "active_input_ratio": 0.0 if total_inputs == 0 else active_inputs / total_inputs,
                }
            )
        return hook

    def make_spike_hook(name):
        def hook(mod, x, y):
            spike_tensor = y[0].detach()
            spike_count = spike_tensor.sum().item()
            spike_rows.append(
                {
                    "layer": name,
                    "spike_count": float(spike_count),
                    "num_neuron_updates": float(spike_tensor.numel()),
                    "spike_rate": 0.0 if spike_tensor.numel() == 0 else spike_count / spike_tensor.numel(),
                }
            )
        return hook

    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            handles.append(module.register_forward_hook(make_synaptic_hook(name, module)))
        if isinstance(module, SPIKE_CELL_TYPES):
            handles.append(module.register_forward_hook(make_spike_hook(name)))

    was_training = model.training
    model.eval()
    with torch.no_grad():
        _ = model(input_tensor)
    model.train(was_training)

    for handle in handles:
        handle.remove()

    synaptic_df = pd.DataFrame(synaptic_rows)
    if not synaptic_df.empty:
        synaptic_df["dense_macs_fmt"] = synaptic_df["dense_macs"].map(format_count)
        synaptic_df["effective_macs_fmt"] = synaptic_df["effective_macs"].map(format_count)

    spike_df = pd.DataFrame(spike_rows)
    return synaptic_df, spike_df


## Real EEG profile from subject 1

For real data, this notebook uses:
- `/scratch/snn/utils/eegmmidb_slice_norm`

For each subject-1 trial it reports:
- THOP dense MACs
- effective synaptic MACs from actual nonzero activity
- total spikes emitted by the SNN blocks during the forward pass


In [6]:
real_data_root = Path(REPO_ROOT / 'dataset' / 'eegmmidb_slice_norm')

if not real_data_root.exists():
    print(f"Real EEG dataset not found at: {real_data_root}")
    real_trial_df = None
    real_summary = None
    subject_synaptic_df = None
    subject_spike_df = None
else:
    ds_params = {
        "base_route": str(real_data_root) + "/",
        "subject_id_list": [SUBJECT_ID],
        "start_ts": 0,
        "end_ts": 161,
        "window_ts": 160,
        "overlap_ts": 0,
        "use_imagery": False,
        "transform": ToTensor(),
    }

    subject_dataset = EEGDataset2DLeftRight(**ds_params)
    print(f"Loaded {len(subject_dataset)} trials for subject {SUBJECT_ID}.")

    trial_rows = []
    synaptic_trials = []
    spike_trials = []

    for trial_idx in range(len(subject_dataset)):
        eeg_trial, label = subject_dataset[trial_idx]
        trial_input = eeg_trial.unsqueeze(0).to(device)

        dense_macs_thop, _ = profile_once(trial_input)
        synaptic_df, spike_df = trace_activity_and_effective_macs(trial_input)

        total_effective_synaptic_macs = synaptic_df["effective_macs"].sum()
        total_spikes = spike_df["spike_count"].sum()
        input_nonzero = trial_input.ne(0).sum().item()
        input_numel = trial_input.numel()

        trial_rows.append(
            {
                "trial_idx": trial_idx,
                "label": int(label),
                "shape": str(tuple(trial_input.shape)),
                "dense_macs_thop": float(dense_macs_thop),
                "dense_macs_fmt": format_count(dense_macs_thop),
                "effective_synaptic_macs": float(total_effective_synaptic_macs),
                "effective_synaptic_macs_fmt": format_count(total_effective_synaptic_macs),
                "total_spikes": float(total_spikes),
                "input_nonzero_ratio": 0.0 if input_numel == 0 else input_nonzero / input_numel,
            }
        )

        synaptic_df = synaptic_df.copy()
        synaptic_df["trial_idx"] = trial_idx
        synaptic_trials.append(synaptic_df)

        spike_df = spike_df.copy()
        spike_df["trial_idx"] = trial_idx
        spike_trials.append(spike_df)

    real_trial_df = pd.DataFrame(trial_rows)
    subject_synaptic_df = pd.concat(synaptic_trials, ignore_index=True)
    subject_spike_df = pd.concat(spike_trials, ignore_index=True)

    real_summary = pd.DataFrame([
        {
            "input": f"subject_{SUBJECT_ID}_average",
            "num_trials": int(len(real_trial_df)),
            "avg_dense_macs_thop": float(real_trial_df["dense_macs_thop"].mean()),
            "avg_dense_macs_fmt": format_count(real_trial_df["dense_macs_thop"].mean()),
            "avg_effective_synaptic_macs": float(real_trial_df["effective_synaptic_macs"].mean()),
            "avg_effective_synaptic_macs_fmt": format_count(real_trial_df["effective_synaptic_macs"].mean()),
            "avg_total_spikes": float(real_trial_df["total_spikes"].mean()),
            "dense_mac_values": int(real_trial_df["dense_macs_thop"].nunique()),
            "effective_mac_values": int(real_trial_df["effective_synaptic_macs"].nunique()),
        }
    ])
    real_summary


Loaded 45 trials for subject 1.


In [7]:
if subject_synaptic_df is not None:
    subject_synaptic_df.groupby(["layer", "module_type"], as_index=False).agg(
        avg_dense_macs=("dense_macs", "mean"),
        avg_effective_macs=("effective_macs", "mean"),
        avg_active_input_ratio=("active_input_ratio", "mean"),
    ).assign(
        avg_dense_macs_fmt=lambda df: df["avg_dense_macs"].map(format_count),
        avg_effective_macs_fmt=lambda df: df["avg_effective_macs"].map(format_count),
    ).sort_values("avg_dense_macs", ascending=False).reset_index(drop=True)


In [8]:
if subject_spike_df is not None:
    subject_spike_df.groupby("layer", as_index=False).agg(
        avg_spike_count=("spike_count", "mean"),
        avg_spike_rate=("spike_rate", "mean"),
    ).sort_values("avg_spike_count", ascending=False).reset_index(drop=True)


In [9]:
subject_synaptic_df.head()

,layer,module_type,dense_macs,effective_macs,savings_ratio,active_inputs,total_inputs,active_input_ratio,dense_macs_fmt,effective_macs_fmt,trial_idx
0,snn.conv1.psp_func,Conv2d,41472.0,31104.0,0.250000,64.0,110.0,0.581818,41.472K,31.104K,0
1,snn.conv2.psp_func,Conv2d,3096576.0,999936.0,0.677083,1533.0,4608.0,0.332682,3.097M,999.936K,0
2,snn.conv3.psp_func,Conv2d,294912.0,113920.0,0.613715,445.0,1152.0,0.386285,294.912K,113.920K,0
3,snn.temp_conv1.psp_func_list.2,Linear,65536.0,25856.0,0.605469,101.0,256.0,0.394531,65.536K,25.856K,0
4,snn.rec1.rec_func,Linear,65536.0,0.0,1.000000,0.0,256.0,0.000000,65.536K,0.000B,0


In [10]:
# Mean effective/dense MACs per synaptic module across all trials
mean_synaptic_per_module = (
    subject_synaptic_df.groupby(["layer", "module_type"], as_index=False)
    .agg(
        mean_dense_macs=("dense_macs", "mean"),
        mean_effective_macs=("effective_macs", "mean"),
        mean_savings_ratio=("savings_ratio", "mean"),
        mean_active_input_ratio=("active_input_ratio", "mean"),
    )
    .assign(
        mean_dense_macs_fmt=lambda df: df["mean_dense_macs"].map(format_count),
        mean_effective_macs_fmt=lambda df: df["mean_effective_macs"].map(format_count),
    )
    .sort_values("mean_dense_macs", ascending=False)
    .reset_index(drop=True)
)


mean_synaptic_per_module


,layer,module_type,mean_dense_macs,mean_effective_macs,mean_savings_ratio,mean_active_input_ratio,mean_dense_macs_fmt,mean_effective_macs_fmt
0,snn.conv2.psp_func,Conv2d,3096576.0,1.197741e+06,0.613205,0.381544,3.097M,1.198M
1,snn.conv3.psp_func,Conv2d,294912.0,1.082291e+05,0.633012,0.366988,294.912K,108.229K
2,snn.temp_conv1.psp_func_list.1,Linear,65536.0,2.610492e+04,0.601671,0.398329,65.536K,26.105K
3,snn.fc1.psp_func,Linear,65536.0,2.545764e+04,0.611547,0.388453,65.536K,25.458K
4,snn.rec1.rec_func,Linear,65536.0,2.530272e+04,0.613911,0.386089,65.536K,25.303K
5,snn.temp_conv1.psp_func_list.2,Linear,65536.0,2.610489e+04,0.601671,0.398329,65.536K,26.105K
6,snn.temp_conv1.psp_func_list.0,Linear,65536.0,2.610419e+04,0.601682,0.398318,65.536K,26.104K
7,snn.conv1.psp_func,Conv2d,41472.0,3.110400e+04,0.250000,0.581818,41.472K,31.104K
8,snn.fc2,Linear,512.0,2.063714e+02,0.596931,0.403069,512.000B,206.371B


In [11]:
mean_synaptic_per_module['mean_effective_macs'].sum()

np.float64(1466354.887224677)

In [12]:
# Mean spike count per spiking module across all trials
mean_spikes_per_module = (
    subject_spike_df.groupby("layer", as_index=False)
    .agg(
        mean_spike_count=("spike_count", "mean"),
        mean_spike_rate=("spike_rate", "mean"),
        mean_neuron_updates=("num_neuron_updates", "mean"),
    )
    .sort_values("mean_spike_count", ascending=False)
    .reset_index(drop=True)
)

mean_spikes_per_module


,layer,mean_spike_count,mean_spike_rate,mean_neuron_updates
0,snn.conv1,1758.154167,0.381544,4608.0
1,snn.conv2,1403.264861,0.261024,5376.0
2,snn.fc1,103.185694,0.403069,256.0
3,snn.conv3,101.972222,0.398329,256.0
4,snn.rec1,99.443889,0.388453,256.0
5,snn.temp_conv1,92.967639,0.363155,256.0


In [13]:
mean_spikes_per_module['mean_spike_count'].sum() * 160

np.float64(569438.1555555556)

In [14]:
mean_total_network_spikes = (
    subject_spike_df.groupby("trial_idx", as_index=False)["spike_count"]
    .sum()["spike_count"]
    .mean()
)

print(mean_total_network_spikes)


569438.1555555556


In [15]:
total_network_spikes_per_trial = (
    subject_spike_df.groupby("trial_idx", as_index=False)
    .agg(total_spike_count=("spike_count", "sum"))
)

total_network_spikes_per_trial


,trial_idx,total_spike_count
0,0,560799.0
1,1,552920.0
2,2,569647.0
3,3,554109.0
4,4,583065.0
5,5,561401.0
6,6,573322.0
7,7,576705.0
8,8,615881.0
9,9,552366.0


In [16]:
neurons_per_timestep = {
    "snn.conv1": 64 * 8 * 9,
    "snn.conv2": 128 * 6 * 7,
    "snn.conv3": 256,
    "snn.temp_conv1": 256,
    "snn.rec1": 256,
    "snn.fc1": 256,
}

print(neurons_per_timestep)
print("Total:", sum(neurons_per_timestep.values()))


{'snn.conv1': 4608, 'snn.conv2': 5376, 'snn.conv3': 256, 'snn.temp_conv1': 256, 'snn.rec1': 256, 'snn.fc1': 256}
Total: 11008
